# 1. Escolha do Município e Obtenção dos Dados:

In [ ]:
import folium
import pandas as pd
import numpy as np

from folium.plugins import Fullscreen, MiniMap
from branca.colormap import LinearColormap

# Leitura dos DataFrames de Ouro Branco (Votação das Sessões Eleitorais e Local de Votação dos Eleitorados):
df_voting_session = pd.read_csv('votacao_secao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')
df_electorate_voting_location = pd.read_csv('eleitorado_local_votacao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')

display(df_voting_session.head())

# 2. Limpeza, Filtragem e Unificação de Dados:

In [29]:
# Adquirindo a localização das sessões eleitorais:

# Padronizando as chaves geográficas, selecionando as colunas para obter a localização de cada seção nas zonas eleitorais:
df_location = df_electorate_voting_location[
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_LATITUDE', 'NR_LONGITUDE']
    ].drop_duplicates(subset=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'])

# Convertendo as coordenadas para valores numéricos:
df_location['NR_LATITUDE'] = df_location['NR_LATITUDE'].astype(str).str.replace(',', '.')
df_location['NR_LONGITUDE'] = df_location['NR_LONGITUDE'].astype(str).str.replace(',', '.')

df_location['NR_LATITUDE'] = pd.to_numeric(df_location['NR_LATITUDE'], errors='coerce')
df_location['NR_LONGITUDE'] = pd.to_numeric(df_location['NR_LONGITUDE'], errors='coerce')

# Removendo coordenadas nulas ou com o valor -1:
df_location = df_location.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE'])
df_location = df_location[(df_location['NR_LATITUDE'] != -1) & (df_location['NR_LONGITUDE'] != -1)]

center = [-20.5233, -43.6914]

# DataFrame com todos os dados que serão utilizados:
df = df_voting_session[['NR_TURNO', 'CD_ELEICAO', 'NR_ZONA', 'NR_SECAO', 'CD_CARGO', 
                        'DS_CARGO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'QT_VOTOS', 'NM_LOCAL_VOTACAO', 'DS_LOCAL_VOTACAO_ENDERECO']].copy()

# Fazendo o merge das colunas 'NR_LATITUDE' e 'NR_LONGITUDE' no DataFrame:
df = df.merge(df_location, on=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'], how='left')

# Agrupando a contagem total de votos por candidato em cada seção eleitoral nos dois turnos:
total_vote_count_grouped = df.groupby(
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE'],
     as_index=False
    )['QT_VOTOS'].sum()

# 3. Mapeamento Graduado do 1º Turno:

In [34]:
# Filtrando os dados do primeiro turno:
df_1t = total_vote_count_grouped[
    total_vote_count_grouped['NR_TURNO'].astype(str) == '1'
].copy()

# Removendo votos brancos e nulos:
df_1t = df_1t[
    ~df_1t['NM_VOTAVEL'].str.contains('BRANCO|NULO', case=False)
].copy()

total_vote_candidates = (
    df_1t.groupby('NM_VOTAVEL')['QT_VOTOS']
    .sum()
    .sort_values(ascending=False)
)

candidates = total_vote_candidates.head(5).index.tolist()

display(total_vote_candidates.head(5))

# Organizando os votos por seção eleitoral e por candidato:
votes_1t = df_1t.pivot_table(
        index=['NR_ZONA', 'NR_SECAO'],
        columns='NM_VOTAVEL',
        values='QT_VOTOS',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

votes_1t.columns.name = None

# Adicionando as informações de localização de cada seção:
df_info_location = df_1t[
    ['NR_ZONA', 'NR_SECAO', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE']
     ].drop_duplicates(subset=['NR_ZONA', 'NR_SECAO'])

votes_1t = votes_1t.merge(df_info_location, on=['NR_ZONA', 'NR_SECAO'], how='left', validate='one_to_one')

# Calculando o total de votos válidos por seção:
candidates_columns = [c for c in votes_1t.columns if c in df_1t['NM_VOTAVEL'].unique()]

votes_1t['TOTAL_VALIDOS'] = votes_1t[candidates_columns].sum(axis=1)

# Calculando o percentual de votos de cada candidato por seção:
for candidate in candidates:
    votes_1t[f'PERCENTUAL_{candidate}'] = np.where(votes_1t['TOTAL_VALIDOS'] > 0, votes_1t[candidate] / votes_1t['TOTAL_VALIDOS'] * 100, 0)

display(votes_1t.head())

palettes = [
    '#272D97',
    '#D60828',
    '#4C8790',
    '#D8942F',
    '#A0A00B'
]

# Associando uma cor aos candidatos:
candidates_palettes = {
    candidate: palettes[i % len(palettes)]
    for i, candidate in enumerate(candidates)
}

def create_map_by_candidate(df, candidate, palette):
    """Função utilizada para criar um mapa para cada candidato, utilizando a latitude e longitude das seções eleitorais."""

    df_map = df.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE']).copy()
    df_map = df_map[(df_map['NR_LATITUDE'] != -1) & (df_map['NR_LONGITUDE'] != -1)]

    map_1t = folium.Map(location=center, zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors')

    scale = LinearColormap(colors=['#ffffff', palette], vmin=0, vmax=100)
    scale.caption = f'Percentual de votos - {candidate} (%)'

    # Iterando as seções eleitorais:
    for _, row in df_map.iterrows():
        percentage = row[f'PERCENTUAL_{candidate}']

        folium.CircleMarker(
            location=[row['NR_LATITUDE'], row['NR_LONGITUDE']],
            radius=8,
            color=palette,
            weight=1,
            fill=True,
            fill_color=scale(percentage),
            fill_opacity=0.85,
        ).add_to(map_1t)
    scale.add_to(map_1t)

    MiniMap(toggle_display=True).add_to(map_1t)

    return map_1t

maps_1t = {}

for candidate in candidates:
    palette = candidates_palettes[candidate]

    map_1t = create_map_by_candidate(votes_1t, candidate, palette)
    maps_1t[candidate] = map_1t

    filename = candidate.lower().replace(" ", "_")
    output_path = f"mapa_1t_{filename}.html"

    map_1t.save(output_path)

NM_VOTAVEL
FLAVIO NANTES BOLSONARO              10891
LUIZ INÁCIO LULA DA SILVA            10187
AUGUSTO JORGE CURY                    1043
RENAN ANTONIO FERREIRA DOS SANTOS      619
RONALDO RAMOS CAIADO                   461
Name: QT_VOTOS, dtype: int64

,NR_ZONA,NR_SECAO,AUGUSTO JORGE CURY,CLARIANA ZACARKIM BARAO,FLAVIO NANTES BOLSONARO,HERTZ DA CONCEICAO DIAS,LEONARDO ALVES DE ARAUJO,LUIZ INÁCIO LULA DA SILVA,RENAN ANTONIO FERREIRA DOS SANTOS,ROMEU ZEMA NETO,...,NM_LOCAL_VOTACAO,DS_LOCAL_VOTACAO_ENDERECO,NR_LATITUDE,NR_LONGITUDE,TOTAL_VALIDOS,PERCENTUAL_FLAVIO NANTES BOLSONARO,PERCENTUAL_LUIZ INÁCIO LULA DA SILVA,PERCENTUAL_AUGUSTO JORGE CURY,PERCENTUAL_RENAN ANTONIO FERREIRA DOS SANTOS,PERCENTUAL_RONALDO RAMOS CAIADO
0,88,138,7,0,102,0,0,103,7,10,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,233,43.776824,44.206009,3.004292,3.004292,1.716738
1,88,139,10,0,105,0,0,92,9,4,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,229,45.851528,40.174672,4.366812,3.930131,3.930131
2,88,140,6,0,107,0,0,95,4,2,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,222,48.198198,42.792793,2.702703,1.801802,2.702703
3,88,141,13,0,135,0,0,110,6,2,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,271,49.815498,40.590406,4.797048,2.214022,1.845018
4,88,142,13,0,123,0,0,101,5,4,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,253,48.616601,39.920949,5.138340,1.976285,2.766798


# 4. Análise do 2º Turno e Comparação Espacial:

In [50]:
# Filtrando os dados do segundo turno:
df_2t = total_vote_count_grouped[
    total_vote_count_grouped['NR_TURNO'].astype(str) == '2'
].copy()

# Removendo votos brancos e nulos:
df_2t = df_2t[
    ~df_2t['NM_VOTAVEL'].str.contains('BRANCO|NULO', case=False)
].copy()

total_vote_candidates = (
    df_2t.groupby('NM_VOTAVEL')['QT_VOTOS']
    .sum()
    .sort_values(ascending=False)
)

candidates = total_vote_candidates.head(2).index.tolist()

display(total_vote_candidates)

# Organizando os votos por seção eleitoral e por candidato:
votes_2t = df_2t.pivot_table(
        index=['NR_ZONA', 'NR_SECAO'],
        columns='NM_VOTAVEL',
        values='QT_VOTOS',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

votes_2t.columns.name = None

# Adicionando as informações de localização de cada seção:
df_info_location = df_2t[
    ['NR_ZONA', 'NR_SECAO', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE']
     ].drop_duplicates(subset=['NR_ZONA', 'NR_SECAO'])

votes_2t = votes_2t.merge(df_info_location, on=['NR_ZONA', 'NR_SECAO'], how='left', validate='one_to_one')

# Calculando o total de votos válidos por seção:
candidates_columns = [c for c in votes_2t.columns if c in df_2t['NM_VOTAVEL'].unique()]

votes_2t['TOTAL_VALIDOS'] = votes_2t[candidates_columns].sum(axis=1)

# Calculando o percentual de votos dos candidatos por seção:
for candidate in candidates:
    votes_2t[f'PERCENTUAL_{candidate}'] = np.where(votes_2t['TOTAL_VALIDOS'] > 0, votes_2t[candidate] / votes_2t['TOTAL_VALIDOS'] * 100, 0)

# Calculando a diferença percentual dos candidatos:
votes_2t['DIFERENCA_PERCENTUAL'] = (
    votes_2t[f'PERCENTUAL_{candidates[0]}'] -
    votes_2t[f'PERCENTUAL_{candidates[1]}']
)

# Identificando o candidato vencedor por seção:
votes_2t['CANDIDATO_LIDER'] = np.select([votes_2t['DIFERENCA_PERCENTUAL'] > 0, votes_2t['DIFERENCA_PERCENTUAL'] < 0], 
                                        [candidates[0], candidates[1]], 
                                        default='Empate'
                                        )
display(votes_2t.head())

palettes = [
    '#272D97',
    '#5265C5',
    '#F2A0A0', 
    '#D60828'
]

# Criando o mapa para os candidatos do segundo turno:
df_map = votes_2t.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE']).copy()
df_map = df_map[(df_map['NR_LATITUDE'] != -1) & (df_map['NR_LONGITUDE'] != -1)]

map_2t = folium.Map(location=center, zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', 
                    attr='&copy; OpenStreetMap contributors')

diverging_scale = LinearColormap(colors=palettes, index=[-100, -0.01, 0.01, 100], vmin=-100, vmax=100)
diverging_scale.caption = (f'Diferença percentual - {candidates[0]} (+) / {candidates[1]} (-)')

# Iterando as seções eleitorais:
for _, row in df_map.iterrows():
        percentage_difference = row['DIFERENCA_PERCENTUAL']

        if percentage_difference == 0:
            palette = '#FFFFFF'

        else:
            palette = diverging_scale(percentage_difference)
        
        tooltip = folium.Tooltip(
            f"""
            <div style="font-family: Arial; font-size: 12px;">
                <b>Zona:</b> {row['NR_ZONA']}<br>
                <b>Seção:</b> {row['NR_SECAO']}<br>
                <b>Local:</b> {row['NM_LOCAL_VOTACAO']}<br>
                <hr>
                <b>{candidates[0]}:</b> {row[f'PERCENTUAL_{candidates[0]}']:.2f}%<br>
                <b>{candidates[1]}:</b> {row[f'PERCENTUAL_{candidates[1]}']:.2f}%<br>
                <hr>
                <b>Candidato líder:</b> {row['CANDIDATO_LIDER']}<br>
                <b>Diferença:</b> {abs(percentage_difference):.2f} p.p.
            </div>
            """,
            sticky=True
        )


        folium.CircleMarker(
            location=[row['NR_LATITUDE'], row['NR_LONGITUDE']],
            radius=20,
            color=palette,
            weight=1,
            fill=True,
            fill_color=palette,
            fill_opacity=0.85,
            tooltip=tooltip
        ).add_to(map_2t)
diverging_scale.add_to(map_2t)

MiniMap(toggle_display=True).add_to(map_2t)

filename = candidate.lower().replace(" ", "_")
output_path = f"mapa_2t.html"

map_2t.save(output_path)

NM_VOTAVEL
LUIZ INÁCIO LULA DA SILVA    12737
FLAVIO NANTES BOLSONARO      11888
Name: QT_VOTOS, dtype: int64

,NR_ZONA,NR_SECAO,FLAVIO NANTES BOLSONARO,LUIZ INÁCIO LULA DA SILVA,NM_LOCAL_VOTACAO,DS_LOCAL_VOTACAO_ENDERECO,NR_LATITUDE,NR_LONGITUDE,TOTAL_VALIDOS,PERCENTUAL_LUIZ INÁCIO LULA DA SILVA,PERCENTUAL_FLAVIO NANTES BOLSONARO,DIFERENCA_PERCENTUAL,CANDIDATO_LIDER
0,88,138,115,125,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,240,52.083333,47.916667,4.166667,LUIZ INÁCIO LULA DA SILVA
1,88,139,115,111,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,226,49.115044,50.884956,-1.769912,FLAVIO NANTES BOLSONARO
2,88,140,120,109,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,229,47.598253,52.401747,-4.803493,FLAVIO NANTES BOLSONARO
3,88,141,147,145,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,292,49.657534,50.342466,-0.684932,FLAVIO NANTES BOLSONARO
4,88,142,142,142,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,284,50.000000,50.000000,0.000000,Empate


In [ ]:
# Filtrando os dados do primeiro turno:
df_1t = total_vote_count_grouped[
    total_vote_count_grouped['NR_TURNO'].astype(str) == '1'
].copy()

# Removendo votos brancos e nulos:
df_1t = df_1t[
    ~df_1t['NM_VOTAVEL'].str.contains('BRANCO|NULO', case=False)
].copy()

total_vote_candidates = (
    df_1t.groupby('NM_VOTAVEL')['QT_VOTOS']
    .sum()
    .sort_values(ascending=False)
)

candidates = total_vote_candidates.head(5).index.tolist()

display(total_vote_candidates.head(5))

# Organizando os votos por seção eleitoral e por candidato:
votes_1t = df_1t.pivot_table(
        index=['NR_ZONA', 'NR_SECAO'],
        columns='NM_VOTAVEL',
        values='QT_VOTOS',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

votes_1t.columns.name = None

# Adicionando as informações de localização de cada seção:
df_info_location = df_1t[
    ['NR_ZONA', 'NR_SECAO', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE']
     ].drop_duplicates(subset=['NR_ZONA', 'NR_SECAO'])

votes_1t = votes_1t.merge(df_info_location, on=['NR_ZONA', 'NR_SECAO'], how='left', validate='one_to_one')

# Calculando o total de votos válidos por seção:
candidates_columns = [c for c in votes_1t.columns if c in df_1t['NM_VOTAVEL'].unique()]

votes_1t['TOTAL_VALIDOS'] = votes_1t[candidates_columns].sum(axis=1)

# Calculando o percentual de votos de cada candidato por seção:
for candidate in candidates:
    votes_1t[f'PERCENTUAL_{candidate}'] = np.where(votes_1t['TOTAL_VALIDOS'] > 0, votes_1t[candidate] / votes_1t['TOTAL_VALIDOS'] * 100, 0)

display(votes_1t.head())

palettes = [
    '#272D97',
    '#D60828',
    '#4C8790',
    '#D8942F',
    '#A0A00B'
]

# Associando uma cor aos candidatos:
candidates_palettes = {
    candidate: palettes[i % len(palettes)]
    for i, candidate in enumerate(candidates)
}

def create_map_by_candidate(df, candidate, palette):
    """Função utilizada para criar um mapa para cada candidato, utilizando a latitude e longitude das seções eleitorais."""

    df_map = df.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE']).copy()
    df_map = df_map[(df_map['NR_LATITUDE'] != -1) & (df_map['NR_LONGITUDE'] != -1)]

    map_1t = folium.Map(location=center, zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors')

    scale = LinearColormap(colors=['#ffffff', palette], vmin=0, vmax=100)
    scale.caption = f'Percentual de votos - {candidate} (%)'

    # Iterando as seções eleitorais:
    for _, row in df_map.iterrows():
        percentage = row[f'PERCENTUAL_{candidate}']

        folium.CircleMarker(
            location=[row['NR_LATITUDE'], row['NR_LONGITUDE']],
            radius=8,
            color=palette,
            weight=1,
            fill=True,
            fill_color=scale(percentage),
            fill_opacity=0.85,
        ).add_to(map_1t)
    scale.add_to(map_1t)

    MiniMap(toggle_display=True).add_to(map_1t)

    return map_1t

maps_1t = {}

for candidate in candidates:
    palette = candidates_palettes[candidate]

    map_1t = create_map_by_candidate(votes_1t, candidate, palette)
    maps_1t[candidate] = map_1t

    filename = candidate.lower().replace(" ", "_")
    output_path = f"mapa_1t_{filename}.html"

    map.save(output_path)

NM_VOTAVEL
FLAVIO NANTES BOLSONARO              10891
LUIZ INÁCIO LULA DA SILVA            10187
AUGUSTO JORGE CURY                    1043
RENAN ANTONIO FERREIRA DOS SANTOS      619
RONALDO RAMOS CAIADO                   461
Name: QT_VOTOS, dtype: int64

,NR_ZONA,NR_SECAO,AUGUSTO JORGE CURY,CLARIANA ZACARKIM BARAO,FLAVIO NANTES BOLSONARO,HERTZ DA CONCEICAO DIAS,LEONARDO ALVES DE ARAUJO,LUIZ INÁCIO LULA DA SILVA,RENAN ANTONIO FERREIRA DOS SANTOS,ROMEU ZEMA NETO,...,NM_LOCAL_VOTACAO,DS_LOCAL_VOTACAO_ENDERECO,NR_LATITUDE,NR_LONGITUDE,TOTAL_VALIDOS,PERCENTUAL_FLAVIO NANTES BOLSONARO,PERCENTUAL_LUIZ INÁCIO LULA DA SILVA,PERCENTUAL_AUGUSTO JORGE CURY,PERCENTUAL_RENAN ANTONIO FERREIRA DOS SANTOS,PERCENTUAL_RONALDO RAMOS CAIADO
0,88,138,7,0,102,0,0,103,7,10,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,233,43.776824,44.206009,3.004292,3.004292,1.716738
1,88,139,10,0,105,0,0,92,9,4,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,229,45.851528,40.174672,4.366812,3.930131,3.930131
2,88,140,6,0,107,0,0,95,4,2,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,222,48.198198,42.792793,2.702703,1.801802,2.702703
3,88,141,13,0,135,0,0,110,6,2,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,271,49.815498,40.590406,4.797048,2.214022,1.845018
4,88,142,13,0,123,0,0,101,5,4,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,253,48.616601,39.920949,5.138340,1.976285,2.766798


In [ ]:
# Filtrando os dados do primeiro turno:
df_1t = total_vote_count_grouped[
    total_vote_count_grouped['NR_TURNO'].astype(str) == '1'
].copy()

# Removendo votos brancos e nulos:
df_1t = df_1t[
    ~df_1t['NM_VOTAVEL'].str.contains('BRANCO|NULO', case=False)
].copy()

total_vote_candidates = (
    df_1t.groupby('NM_VOTAVEL')['QT_VOTOS']
    .sum()
    .sort_values(ascending=False)
)

candidates = total_vote_candidates.head(5).index.tolist()

display(total_vote_candidates.head(5))

# Organizando os votos por seção eleitoral e por candidato:
votes_1t = df_1t.pivot_table(
        index=['NR_ZONA', 'NR_SECAO'],
        columns='NM_VOTAVEL',
        values='QT_VOTOS',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

votes_1t.columns.name = None

# Adicionando as informações de localização de cada seção:
df_info_location = df_1t[
    ['NR_ZONA', 'NR_SECAO', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE']
     ].drop_duplicates(subset=['NR_ZONA', 'NR_SECAO'])

votes_1t = votes_1t.merge(df_info_location, on=['NR_ZONA', 'NR_SECAO'], how='left', validate='one_to_one')

# Calculando o total de votos válidos por seção:
candidates_columns = [c for c in votes_1t.columns if c in df_1t['NM_VOTAVEL'].unique()]

votes_1t['TOTAL_VALIDOS'] = votes_1t[candidates_columns].sum(axis=1)

# Calculando o percentual de votos de cada candidato por seção:
for candidate in candidates:
    votes_1t[f'PERCENTUAL_{candidate}'] = np.where(votes_1t['TOTAL_VALIDOS'] > 0, votes_1t[candidate] / votes_1t['TOTAL_VALIDOS'] * 100, 0)

display(votes_1t.head())

palettes = [
    '#272D97',
    '#D60828',
    '#4C8790',
    '#D8942F',
    '#A0A00B'
]

# Associando uma cor aos candidatos:
candidates_palettes = {
    candidate: palettes[i % len(palettes)]
    for i, candidate in enumerate(candidates)
}

def create_map_by_candidate(df, candidate, palette):
    """Função utilizada para criar um mapa para cada candidato, utilizando a latitude e longitude das seções eleitorais."""

    df_map = df.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE']).copy()
    df_map = df_map[(df_map['NR_LATITUDE'] != -1) & (df_map['NR_LONGITUDE'] != -1)]

    center = [-20.5233, -43.6914]

    map = folium.Map(location=center, zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors')

    scale = LinearColormap(colors=['#ffffff', palette], vmin=0, vmax=100)
    scale.caption = f'Percentual de votos - {candidate} (%)'

    for _, row in df_map.iterrows():

        percentage = row[f'PERCENTUAL_{candidate}']

        folium.CircleMarker(
            location=[row['NR_LATITUDE'], row['NR_LONGITUDE']],
            radius=8,
            color=palette,
            weight=1,
            fill=True,
            fill_color=scale(percentage),
            fill_opacity=0.85,
        ).add_to(map)
    scale.add_to(map)

    MiniMap(toggle_display=True).add_to(map)

    return map

maps_1t = {}

for candidate in candidates:
    palette = candidates_palettes[candidate]

    map = create_map_by_candidate(votes_1t, candidate, palette)
    maps_1t[candidate] = map

    filename = candidate.lower().replace(" ", "_")
    output_path = f"mapa_1t_{filename}.html"

    map.save(output_path)

NM_VOTAVEL
FLAVIO NANTES BOLSONARO              10891
LUIZ INÁCIO LULA DA SILVA            10187
AUGUSTO JORGE CURY                    1043
RENAN ANTONIO FERREIRA DOS SANTOS      619
RONALDO RAMOS CAIADO                   461
Name: QT_VOTOS, dtype: int64

,NR_ZONA,NR_SECAO,AUGUSTO JORGE CURY,CLARIANA ZACARKIM BARAO,FLAVIO NANTES BOLSONARO,HERTZ DA CONCEICAO DIAS,LEONARDO ALVES DE ARAUJO,LUIZ INÁCIO LULA DA SILVA,RENAN ANTONIO FERREIRA DOS SANTOS,ROMEU ZEMA NETO,...,NM_LOCAL_VOTACAO,DS_LOCAL_VOTACAO_ENDERECO,NR_LATITUDE,NR_LONGITUDE,TOTAL_VALIDOS,PERCENTUAL_FLAVIO NANTES BOLSONARO,PERCENTUAL_LUIZ INÁCIO LULA DA SILVA,PERCENTUAL_AUGUSTO JORGE CURY,PERCENTUAL_RENAN ANTONIO FERREIRA DOS SANTOS,PERCENTUAL_RONALDO RAMOS CAIADO
0,88,138,7,0,102,0,0,103,7,10,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,233,43.776824,44.206009,3.004292,3.004292,1.716738
1,88,139,10,0,105,0,0,92,9,4,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,229,45.851528,40.174672,4.366812,3.930131,3.930131
2,88,140,6,0,107,0,0,95,4,2,...,SPARTA TÊNIS CLUBE (ANTIGA AEA),"AV. PATRIOTICA ,800",-20.514220,-43.701786,222,48.198198,42.792793,2.702703,1.801802,2.702703
3,88,141,13,0,135,0,0,110,6,2,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,271,49.815498,40.590406,4.797048,2.214022,1.845018
4,88,142,13,0,123,0,0,101,5,4,...,CAMARA MUNICIPAL DE OURO BRANCO,"PRACA SAGRADO CORACOES, 200",-20.512249,-43.722183,253,48.616601,39.920949,5.138340,1.976285,2.766798


In [ ]:
# Criar mapa base
mapa_marcadores = folium.Map(location=[-15.793889, -47.882778], zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')

# Marcador 1: Esplanada dos Ministérios
folium.Marker(
    location=[-15.7981, -47.8756],
    popup="<b>Esplanada dos Ministérios</b><br>Centro do Governo Federal",
    tooltip="Clique para saber mais",
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(mapa_marcadores)

# Marcador 2: Congresso Nacional (Ícone personalizado)
folium.Marker(
    location=[-15.7998, -47.8642],
    popup="<b>Congresso Nacional</b>",
    tooltip="Congresso Nacional",
    icon=folium.Icon(color='green', icon='star')
).add_to(mapa_marcadores)

mapa_marcadores

In [ ]:
mapa_circulos = folium.Map(location=[-15.793889, -47.882778], zoom_start=13, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')

# CircleMarker: Raio fixo em Pixels (útil para pontos agregados)
folium.CircleMarker(
    location=[-15.7981, -47.8756],
    radius=15,
    popup="Área Central (Raio Fixo)",
    color='blue',
    fill=True,
    fill_color='lightblue',
    fill_opacity=0.6
).add_to(mapa_circulos)

# Circle: Raio real em Metros (útil para raio de cobertura)
folium.Circle(
    location=[-15.793889, -47.882778],
    radius=2000, # 2 km de raio
    popup="Zona de Cobertura de 2km",
    color='red',
    fill=True,
    fill_opacity=0.2
).add_to(mapa_circulos)

mapa_circulos

In [ ]:
from folium import plugins
mapa_plugins = folium.Map(location=[-15.793889, -47.882778], zoom_start=11, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')

# Dados simulados de ocorrências/pontos (Latitude, Longitude)
dados_pontos = [
    [-15.7981, -47.8756], [-15.7998, -47.8642], [-15.7800, -47.8900],
    [-15.7850, -47.8850], [-15.8000, -47.8700], [-15.8100, -47.8600],
    [-15.8200, -47.8950], [-15.7700, -47.8800], [-15.7750, -47.8700]
]

# 1. Agrupamento de Marcadores (MarkerCluster)
cluster = plugins.MarkerCluster(name="Agrupamento de Pontos").add_to(mapa_plugins)
for lat_lng in dados_pontos:
    folium.Marker(location=lat_lng).add_to(cluster)

# 2. Mapa de Calor (HeatMap)
plugins.HeatMap(dados_pontos, name="Mapa de Calor").add_to(mapa_plugins)

# Adicionar Controle de Camadas para ligar/desligar
folium.LayerControl().add_to(mapa_plugins)

mapa_plugins

In [ ]:
# 1. Criar mapa principal
mapa_utilitarios = folium.Map(location=[-15.793889, -47.882778], zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')


# 2. Criar a camada de tiles pública explicitamente
camada_mini = folium.TileLayer(
    tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors'
)

# 3. Passar a camada para o MiniMap
minimap = plugins.MiniMap(
    tile_layer=camada_mini,
    toggle_display=True,
    position='bottomright'
)

# 4. Adicionar ao mapa principal
mapa_utilitarios.add_child(minimap)

# Régua de Medição de Distância/Área
mapa_utilitarios.add_child(plugins.MeasureControl(position='topright'))

# Popup para mostrar coordenadas ao clicar
mapa_utilitarios.add_child(folium.LatLngPopup())

mapa_utilitarios

In [ ]:
# 1. Carregar Dados de Desemprego nos EUA
url = 'https://raw.githubusercontent.com/python-visualization/folium/main/examples/data'
df_desemprego = pd.read_csv(f'{url}/US_Unemployment_Oct2012.csv')
geo_json_estados = f'{url}/us-states.json'

print("Primeiras linhas do DataFrame:")
print(df_desemprego.head())

# 2. Criar Mapa Base dos EUA
mapa_choropleth = folium.Map(location=[48, -102], zoom_start=4, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors' )

# 3. Adicionar a Camada Choropleth
folium.Choropleth(
    geo_data=geo_json_estados,
    name='Taxa de Desemprego',
    data=df_desemprego,
    columns=['State', 'Unemployment'],
    key_on='feature.id',            # Chave de ligação no GeoJSON
    fill_color='YlOrRd',            # Escala Amarelo-Laranja-Vermelho
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Taxa de Desemprego (%)'
).add_to(mapa_choropleth)

folium.LayerControl().add_to(mapa_choropleth)

mapa_choropleth

In [ ]:
# Salvar o mapa final como um arquivo HTML estático/interativo
mapa_choropleth.save("mapa_desemprego_eua.html")
print("Mapa salvo com sucesso em 'mapa_desemprego_eua.html'!")

In [ ]:
import folium
import pandas as pd

# 1. Dados de exemplo (Estados e um valor numérico fictício, ex: Pontuação/Índice)
dados_estados = pd.DataFrame({
    'UF': ['SP', 'RJ', 'MG', 'BA', 'RS', 'PR', 'PE', 'CE', 'PA', 'AM'],
    'Valor': [95.5, 82.3, 88.0, 75.1, 84.6, 86.2, 72.4, 70.8, 65.0, 68.3]
})

# 2. URL do GeoJSON com a malha dos Estados do Brasil
url_geojson = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'

# 3. Criar o Mapa Base centralizado no Brasil
mapa_br = folium.Map(location=[-14.2350, -51.9253], zoom_start=4,tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', attr='&copy; OpenStreetMap contributors')

# 4. Adicionar a camada Choropleth
folium.Choropleth(
    geo_data=url_geojson,
    name='Choropleth Brasil',
    data=dados_estados,
    columns=['UF', 'Valor'],
    key_on='feature.properties.sigla',  # A propriedade 'sigla' no GeoJSON contém 'SP', 'RJ', etc.
    fill_color='YlGnBu',                 # Escala de cores (Amarelo, Verde, Azul)
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Índice por Estado'
).add_to(mapa_br)

# Adicionar controle para ligar/desligar camadas
folium.LayerControl().add_to(mapa_br)

mapa_br

In [ ]:
import folium
import pandas as pd

# 1. Dados dos Estados (Exemplo)
dados_estados = pd.DataFrame({
    'UF': ['SP', 'RJ', 'MG', 'BA', 'RS', 'PR', 'PE', 'CE', 'PA', 'AM'],
    'Nome_Estado': ['São Paulo', 'Rio de Janeiro', 'Minas Gerais', 'Bahia',
                   'Rio Grande do Sul', 'Paraná', 'Pernambuco', 'Ceará', 'Pará', 'Amazonas'],
    'Valor': [95.5, 82.3, 88.0, 75.1, 84.6, 86.2, 72.4, 70.8, 65.0, 68.3]
})

# 2. GeoJSON com a malha dos Estados do Brasil
url_geojson = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'

# 3. Criar o Mapa Base
mapa = folium.Map(location=[-14.2350, -51.9253], zoom_start=4,tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', attr='&copy; OpenStreetMap contributors')

# 4. Criar a camada Choropleth para a coloração graduada
choropleth = folium.Choropleth(
    geo_data=url_geojson,
    name='Choropleth Brasil',
    data=dados_estados,
    columns=['UF', 'Valor'],
    key_on='feature.properties.sigla',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Índice por Estado'
).add_to(mapa)

estilo_tooltip = """
    background-color: #2b2b2b;
    color: #ffffff;
    font-family: 'Helvetica Neue', Arial, sans-serif;
    font-size: 13px;
    border-radius: 6px;
    border: 1px solid #444444;
    padding: 8px 12px;
    box-shadow: 3px 3px 10px rgba(0, 0, 0, 0.4);
"""

# Criando o GeoJsonTooltip estilizado
tooltip_customizado = folium.GeoJsonTooltip(
    fields=['name', 'sigla'],             # Propriedades presentes no arquivo GeoJSON
    aliases=['Estado:', 'UF:'],            # Rótulos amigáveis exibidos no balão
    style=estilo_tooltip,                   # CSS aplicado
    sticky=True,                            # Acompanha o ponteiro do mouse
    localize=True                           # Formatação local de números
)
folium.GeoJson(url_geojson, tooltip=tooltip_customizado).add_to(mapa)
mapa

In [ ]:
import folium

# 1. Inicializa o mapa sem nenhum tile padrão
mapa = folium.Map(
    location=[-23.55052, -46.633308],  # São Paulo
    zoom_start=12,
    tiles=None
)

# 2. Define e adiciona as 3 opções de tiles XYZ

# Opção 1: CyclOSM (Estilo ciclístico/topográfico)
url_cyclosm = 'https://{s}.tile-cyclosm.openstreetmap.fr/cyclosm/{z}/{x}/{y}.png'
attr_cyclosm = '&copy; CyclOSM via OpenStreetMap'
folium.TileLayer(
    tiles=url_cyclosm,
    attr=attr_cyclosm,
    name="Ciclovias e Relevo (CyclOSM)",
    control=True
).add_to(mapa)

# Opção 2: Humanitarian OpenStreetMap (Estilo infraestrutura/colorido)
url_humanitarian = 'https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png'
attr_humanitarian = '&copy; OpenStreetMap contributors, Humanitarian OpenStreetMap Team'
folium.TileLayer(
    tiles=url_humanitarian,
    attr=attr_humanitarian,
    name="Humanitário (OSM HOT)",
    control=True
).add_to(mapa)


# 3. Adiciona o controle de camadas (menu interativo)

folium.LayerControl(position="topright", collapsed=False).add_to(mapa)

mapa